# 🤝 Módulo 11 — Orchestration: Handoff

> **Objetivo:** Construir un sistema de atención multi-agente donde el control se transfiere dinámicamente entre especialistas.

📚 **Doc oficial:** [Handoff Orchestration (Python)](https://learn.microsoft.com/en-us/agent-framework/workflows/orchestrations/handoff?pivots=programming-language-python)

## ¿Qué es Handoff Orchestration?

Los agentes están conectados en **topología de malla** — cada uno puede transferir el control
a otro agente especializado según el contexto. No hay un orquestador central que decida.

```
        ┌──────────┐
        │  triage  │
        └──┬─────┬─┘
       ┌───┘     └───┐
  ┌────▼──┐      ┌──▼─────┐
  │ refund │     │ orders │
  └────────┘     └────────┘
```

Ideal para:
- **Customer support** — el agente correcto atiende según la consulta
- Sistemas expertos con dominios bien delimitados
- Cualquier escenario que requiera delegación dinámica

## Handoff vs Agent-as-Tools

| Aspecto | Handoff | Agent-as-Tools |
|---------|---------|----------------|
| Control | Se transfiere completamente | El agente principal retiene el control |
| Ownership | El receptor asume la tarea | El principal sigue siendo responsable |
| Contexto | Conversación completa se transfiere | Sólo lo necesario se pasa al tool |

## API clave

```python
from agent_framework.orchestrations import HandoffBuilder, HandoffAgentUserRequest

workflow = (
    HandoffBuilder(name="support", participants=[triage, refund, order])
    .with_start_agent(triage)
    .build()
)
```


## ⚙️ Setup

In [1]:
# Aseguramos que la raíz del proyecto esté en sys.path para importar `helpers`
import sys
from pathlib import Path

_ROOT = Path.cwd()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from agent_framework import Agent
from helpers.config import create_chat_client
print("✅ Helpers cargados.")


C:\Users\jreyesleger\AppData\Roaming\Python\Python312\site-packages\agent_framework\_skills.py:116: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
C:\Users\jreyesleger\AppData\Roaming\Python\Python312\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


✅ Helpers cargados.


## 1️⃣ Definir tools y agentes especializados

Cada agente tiene tools relevantes a su dominio. El `triage_agent` no tiene tools — sólo enruta.


In [2]:
from typing import Annotated
from agent_framework import tool


@tool(approval_mode="never_require")
def process_refund(
    order_number: Annotated[str, "Número de pedido para procesar el reembolso"],
) -> str:
    """Función simulada para procesar un reembolso para un número de pedido dado."""
    return f"Reembolso procesado exitosamente para el pedido {order_number}."


@tool(approval_mode="never_require")
def check_order_status(
    order_number: Annotated[str, "Número de pedido para verificar el estado"],
) -> str:
    """Función simulada para verificar el estado de un número de pedido dado."""
    return f"El pedido {order_number} está siendo procesado y se enviará en 2 días hábiles."


@tool(approval_mode="never_require")
def process_return(
    order_number: Annotated[str, "Número de pedido para procesar la devolución"],
) -> str:
    """Función simulada para procesar una devolución para un número de pedido dado."""
    return f"Devolución iniciada para el pedido {order_number}. Instrucciones enviadas por correo."


client = create_chat_client()

# require_per_service_call_history_persistence=True es requerido por HandoffBuilder
# para mantener el historial sincronizado durante los handoffs entre agentes.
triage_agent = Agent(
    client=client,
    name="triage_agent",
    description="Agente de triaje que maneja consultas generales.",
    instructions=(
        "Eres el soporte de primera línea. Enruta los problemas de los clientes al agente "
        "especialista apropiado según el problema descrito. Responde en español."
    ),
    require_per_service_call_history_persistence=True,
)

refund_agent = Agent(
    client=client,
    name="refund_agent",
    description="Agente que maneja solicitudes de reembolso.",
    instructions="Procesas solicitudes de reembolso. Responde en español.",
    tools=[process_refund],
    require_per_service_call_history_persistence=True,
)

order_agent = Agent(
    client=client,
    name="order_agent",
    description="Agente que maneja seguimiento de pedidos y envíos.",
    instructions="Manejas consultas de pedidos y envíos. Responde en español.",
    tools=[check_order_status],
    require_per_service_call_history_persistence=True,
)

return_agent = Agent(
    client=client,
    name="return_agent",
    description="Agente que maneja procesamiento de devoluciones.",
    instructions="Gestionas solicitudes de devolución de productos. Responde en español.",
    tools=[process_return],
    require_per_service_call_history_persistence=True,
)

print("✅ 4 agentes especializados creados")

✅ 4 agentes especializados creados


## 2️⃣ Construir el workflow con reglas de handoff

Por defecto **todos los agentes pueden delegar a todos**. Usa `add_handoff(from, [to_list])`
para restringir las rutas.

> ℹ️ Aun con reglas custom, todos los agentes siguen conectados (malla) para que compartan
> contexto. Las reglas solo controlan **quién puede iniciar el handoff**.


In [4]:
from agent_framework.orchestrations import HandoffBuilder

workflow = (
    HandoffBuilder(
        name="customer_support_handoff",
        participants=[triage_agent, refund_agent, order_agent, return_agent],
    )
    .with_start_agent(triage_agent)  # triage recibe el input inicial
    # Triage NO puede mandar directo a refund — primero debe pasar por return
    .add_handoff(triage_agent, [order_agent, return_agent])
    # Solo el agente de return puede delegar a refund (devoluciones → reembolso)
    .add_handoff(return_agent, [refund_agent])
    # Todos los especialistas pueden volver a triage
    .add_handoff(order_agent, [triage_agent])
    .add_handoff(return_agent, [triage_agent])
    .add_handoff(refund_agent, [triage_agent])
    .build()
)

print("✅ Workflow de handoff construido con rutas restringidas")


✅ Workflow de handoff construido con rutas restringidas


## 3️⃣ Modo autónomo (sin input humano)

Por defecto el handoff es **interactivo** — si un agente decide NO delegar, el workflow pausa
y pide input al usuario. Para demos no interactivos, activa `with_autonomous_mode()`.

Puntos clave para que funcione bien:

- **`termination_condition`** — define cuándo terminar el workflow (ej. cuando la conversación tiene suficientes mensajes o cuando se detecta una frase clave).
- **`intermediate_output_from`** — marca agentes cuya respuesta es "de paso" (ej. triage solo enruta, no produce output final).
- **`turn_limits`** — evita loops infinitos limitando cuántos turnos autónomos puede hacer cada agente.

In [10]:
# Modo autónomo con termination_condition para un demo limpio.
#
# - termination_condition: termina cuando ya hay >=2 mensajes en la conversación
#   (triage enruta → especialista responde → se detiene)
# - intermediate_output_from: triage solo enruta, su output no es "final"
# - turn_limits: seguro extra para evitar loops infinitos
workflow = (
    HandoffBuilder(
        name="autonomous_support",
        participants=[triage_agent, refund_agent, order_agent, return_agent],
        termination_condition=lambda msgs: len(msgs) >= 2,
    )
    .with_start_agent(triage_agent)
    .with_autonomous_mode(turn_limits={
        triage_agent.name: 1,
        refund_agent.name: 1,
        order_agent.name: 1,
        return_agent.name: 1,
    })
    .build()
)

from agent_framework import AgentResponseUpdate

print("📨 Prompt: 'Process a refund for order 12345'\n")

last_author = None
async for event in workflow.run("Procesa un reembolso para el pedido 12345", stream=True):
    if isinstance(getattr(event, "data", None), AgentResponseUpdate):
        update = event.data
        # Solo mostrar si hay texto (triage puede generar eventos vacíos al hacer handoff)
        if not update.text:
            continue
        if update.author_name != last_author:
            if last_author is not None:
                print()  # salto de línea entre agentes
            print(f"🤖 [{update.author_name}]: ", end="", flush=True)
            last_author = update.author_name
        print(update.text, end="", flush=True)

print("\n\n✅ Demo de modo autónomo completado.")

📨 Prompt: 'Process a refund for order 12345'

🤖 [order_agent]: El estado de la orden 12345 indica que actualmente está en proceso y se enviará en 2 días hábiles.

✅ Demo de modo autónomo completado.


## 4️⃣ Modo interactivo (HITL) — Simulación multi-agente

Simulamos una conversación donde el usuario pide un reembolso y luego pregunta por otro pedido,
forzando **handoffs entre distintos agentes**:

```
👤 "Necesito un reembolso para el pedido 12345"
   → 🤖 triage → handoff → 🤖 refund_agent (procesa)
👤 "Ahora revisa el estado de mi pedido 67890"
   → 🤖 refund_agent → handoff → 🤖 triage → handoff → 🤖 order_agent (responde)
👤 "¡Eso es todo, gracias!"
   → workflow termina
```

In [16]:
from agent_framework import AgentResponseUpdate
from agent_framework.orchestrations import HandoffAgentUserRequest

# Conversación simulada que fuerza handoffs entre distintos agentes
user_messages = [
    "Necesito un reembolso para el pedido 12345",               # → triage → refund_agent
    "Ahora revisa el estado de mi pedido 67890",         # → refund → triage → order_agent
    "¡Eso es todo, gracias!",                            # → terminate
]

workflow = (
    HandoffBuilder(
        name="multi_agent_demo",
        participants=[triage_agent, refund_agent, order_agent, return_agent],
    )
    .with_start_agent(triage_agent)
    .build()
)

# Helper: procesar un stream y mostrar respuestas
async def run_and_show(stream):
    """Consume el stream, imprime respuestas agrupadas por agente, retorna el pending request."""
    pending, last = None, None
    async for event in stream:
        if isinstance(getattr(event, "data", None), AgentResponseUpdate) and event.data.text:
            if event.data.author_name != last:
                if last: print()
                print(f"   🤖 [{event.data.author_name}]: ", end="")
                last = event.data.author_name
            print(event.data.text, end="", flush=True)
        if event.type == "request_info" and isinstance(event.data, HandoffAgentUserRequest):
            pending = event
    if last: print()
    return pending

# Enviar primer mensaje
msg = user_messages[0]
print(f"👤 Usuario: {msg}")
pending = await run_and_show(workflow.run(msg, stream=True))

# Continuar con el resto de mensajes
for msg in user_messages[1:]:
    if not pending:
        break
    print(f"\n👤 Usuario: {msg}")
    responses = {pending.request_id: HandoffAgentUserRequest.create_response(msg)}
    pending = await run_and_show(workflow.run(responses=responses, stream=True))

# Terminar limpiamente si queda un request pendiente
if pending:
    responses = {pending.request_id: HandoffAgentUserRequest.terminate()}
    async for _ in workflow.run(responses=responses, stream=True):
        pass

print("\n✅ Conversación multi-agente completada.")

👤 Usuario: I need a refund for order 12345
   🤖 [refund_agent]: Your refund for order 12345 has been processed successfully. Let me know if there's anything else I can help you with!

👤 Usuario: Now check the status of my order 67890
   🤖 [order_agent]: Order 67890 is currently being processed and is scheduled to ship in 2 business days. Let me know if there's anything else you'd like to know!

👤 Usuario: That's all, thanks!
   🤖 [order_agent]: You're welcome! If you need any further assistance, don't hesitate to reach out. Have a great day! 😊

✅ Conversación multi-agente completada.


## 🎯 Resumen

| Capacidad | API |
|-----------|-----|
| Workflow básico | `HandoffBuilder(name=..., participants=[...]).with_start_agent(triage).build()` |
| Restringir rutas | `.add_handoff(from_agent, [to_agents])` |
| Modo autónomo | `.with_autonomous_mode(turn_limits={...})` |
| Modo interactivo | Detectar `request_info` + `HandoffAgentUserRequest` |
| Responder al request | `HandoffAgentUserRequest.create_response("...")` |
| Terminar workflow | `HandoffAgentUserRequest.terminate()` |

> 📖 Para casos con tool approval mezclado, revisa la sección "Tool Approval in Handoff Workflows" de la [doc oficial](https://learn.microsoft.com/en-us/agent-framework/workflows/orchestrations/handoff?pivots=programming-language-python#advanced-tool-approval-in-handoff-workflows).

**Siguiente módulo →** [Módulo 12 — Orchestration Group Chat](./12_orchestration_groupchat.ipynb)
